# Safety RM + GRPO Training

This notebook implements the reward distillation + GRPO pipeline:

1. **Train a Safety Reward Model (RM)** on the 485 teacher/base preference pairs from `dpo_pairs.jsonl`
2. **Run GRPO** starting from the SFT baseline, using the frozen RM to score responses to adversarial prompts

Output: `outputs/rm/` and `outputs/grpo/` LoRA adapters — to be compared against `outputs/dpo/`.

In [ ]:
# NOTE: do NOT pin transformers here — RewardConfig and GRPOTrainer need newer transformers
# (unlike kaggle_train.ipynb which pins transformers==4.46.3 for SFTTrainer/DPOConfig)
!pip install -q transformers trl>=0.14.0 peft accelerate bitsandbytes datasets

In [ ]:
import os, gc, json, csv, io, shutil, torch, urllib.request
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import PeftModel, LoraConfig, get_peft_model, TaskType
from trl import RewardTrainer, RewardConfig, GRPOTrainer, GRPOConfig

# ── Constants ────────────────────────────────────────────────────────────────
BASE_MODEL_ID    = "meta-llama/Llama-3.2-1B"
REPO_URL         = "https://github.com/202520030411/Distill-Safety-Aligned-Models.git"
SFT_BASELINE_DIR = "repo/outputs/baseline"
DPO_PAIRS_PATH   = "repo/data/dpo_pairs.jsonl"

RM_OUTPUT_DIR   = "/kaggle/working/outputs/rm"
GRPO_OUTPUT_DIR = "/kaggle/working/outputs/grpo"
os.makedirs(RM_OUTPUT_DIR,   exist_ok=True)
os.makedirs(GRPO_OUTPUT_DIR, exist_ok=True)

print("TRL version:", __import__('trl').__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN loaded")

In [ ]:
!git clone {REPO_URL} repo
!echo "Repo cloned. Contents:"
!ls repo/

## Load Data

In [ ]:
# ── Load DPO pairs for RM training ───────────────────────────────────────────
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

pairs = load_jsonl(DPO_PAIRS_PATH)
print(f"Loaded {len(pairs)} preference pairs")
print("Example:", pairs[0]["prompt"][:80])

# Format for RewardTrainer: chosen/rejected as full text
def format_pair(ex):
    prompt = ex["prompt"]
    return {
        "chosen":   f"Human: {prompt}\n\nAssistant: {ex['chosen']}",
        "rejected": f"Human: {prompt}\n\nAssistant: {ex['rejected']}",
    }

rm_ds = Dataset.from_list([format_pair(p) for p in pairs])
split = rm_ds.train_test_split(test_size=0.1, seed=42)
rm_train_ds = split["train"]
rm_eval_ds  = split["test"]
print(f"RM train: {len(rm_train_ds)}  eval: {len(rm_eval_ds)}")

In [ ]:
# ── Load AdvBench harmful prompts for GRPO ────────────────────────────────────
ADVBENCH_URL = (
    "https://raw.githubusercontent.com/llm-attacks/llm-attacks/main"
    "/data/advbench/harmful_behaviors.csv"
)
with urllib.request.urlopen(ADVBENCH_URL) as resp:
    reader = csv.DictReader(io.StringIO(resp.read().decode()))
    advbench_prompts = [row["goal"] for row in reader]

print(f"AdvBench prompts: {len(advbench_prompts)}")
print("Example:", advbench_prompts[0])

## Helper Functions

In [ ]:
REFUSAL_PREFIXES = [
    "I cannot", "I can't", "I'm not able", "I won't", "I'm unable",
    "I don't", "I apologize", "As an AI", "I'm sorry", "I must decline",
    "I'm not going to", "That's not something", "I am not able",
]

def is_refusal(text):
    t = text.lower()
    return any(p.lower() in t for p in REFUSAL_PREFIXES)

def free_memory(*objs):
    for obj in objs:
        del obj
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

QNF4_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

## Stage 1 — Train Safety Reward Model

In [ ]:
print("=" * 60)
print("STAGE 1: Training Safety Reward Model")
print("=" * 60)

rm_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=os.environ["HF_TOKEN"])
rm_tokenizer.pad_token = rm_tokenizer.eos_token
rm_tokenizer.padding_side = "right"

# AutoModelForSequenceClassification adds a single linear head (num_labels=1)
# → outputs a scalar safety score per input
rm_base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_ID,
    num_labels=1,
    quantization_config=QNF4_CONFIG,
    device_map={"": 0},
    token=os.environ["HF_TOKEN"],
)
rm_base.config.pad_token_id = rm_tokenizer.pad_token_id

# Add LoRA for efficient fine-tuning
rm_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)
rm_model = get_peft_model(rm_base, rm_lora)
rm_model.print_trainable_parameters()

In [ ]:
# Loss: -log σ(r_chosen − r_rejected)
# RewardTrainer handles this automatically from the chosen/rejected text fields
rm_config = RewardConfig(
    output_dir=RM_OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    max_length=512,
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    remove_unused_columns=False,
    report_to="none",
)

rm_trainer = RewardTrainer(
    model=rm_model,
    args=rm_config,
    train_dataset=rm_train_ds,
    eval_dataset=rm_eval_ds,
    processing_class=rm_tokenizer,
)
rm_trainer.train()

rm_model.save_pretrained(RM_OUTPUT_DIR)
rm_tokenizer.save_pretrained(RM_OUTPUT_DIR)
print("RM saved to", RM_OUTPUT_DIR)

In [ ]:
# Free ALL RM training objects from GPU before reloading
del rm_trainer
del rm_model
del rm_base
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print("RM training objects freed")

# Reload as frozen inference-only model
rm_frozen_base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_ID,
    num_labels=1,
    quantization_config=QNF4_CONFIG,
    device_map="auto",
    token=os.environ["HF_TOKEN"],
)
rm_frozen_base.config.pad_token_id = rm_tokenizer.pad_token_id
rm_frozen = PeftModel.from_pretrained(rm_frozen_base, RM_OUTPUT_DIR)
rm_frozen.eval()
for p in rm_frozen.parameters():
    p.requires_grad = False

print("Frozen RM ready for reward scoring")

## Stage 2 — GRPO Training

In [ ]:
# ── Reward function: RM score + refusal heuristic ────────────────────────────
# GRPOTrainer calls: reward_func(prompts=..., completions=..., completion_ids=..., **kwargs)
# completions may be plain strings or list-of-dicts (chat format) depending on TRL version

def _extract_text(item):
    """Handle both plain string and chat-format completions."""
    if isinstance(item, str):
        return item
    if isinstance(item, list):
        return " ".join(m.get("content", "") for m in item if isinstance(m, dict))
    return str(item)

def safety_reward_fn(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        prompt_text = _extract_text(prompt)
        completion_text = _extract_text(completion)
        full_text = prompt_text.rstrip() + " " + completion_text.lstrip()

        inputs = rm_tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True,
        ).to(rm_frozen.device)

        with torch.no_grad():
            rm_score = rm_frozen(**inputs).logits[0, 0].item()

        heuristic = 0.5 if is_refusal(completion_text) else -0.5
        rewards.append(rm_score + heuristic)

    return rewards

In [ ]:
# ── GRPO dataset: AdvBench prompts formatted as generation prefixes ───────────
PROMPT_TEMPLATE = "Human: {}\n\nAssistant:"
grpo_dataset = Dataset.from_dict({
    "prompt": [PROMPT_TEMPLATE.format(p) for p in advbench_prompts]
})
print(f"GRPO dataset: {len(grpo_dataset)} prompts")

In [ ]:
print("=" * 60)
print("STAGE 2: GRPO Training")
print("=" * 60)
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# ── Step A: Merge SFT adapter into base, save to disk ────────────────────────
# Free frozen RM first
del rm_frozen, rm_frozen_base, _rm_base
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()

MERGED_SFT_DIR = "/kaggle/working/merged_sft"
print("Merging SFT adapter into base weights (bfloat16)...")
_merge_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map={"": 0},
    token=os.environ["HF_TOKEN"],
)
_merge_base = PeftModel.from_pretrained(_merge_base, SFT_BASELINE_DIR)
_merge_base = _merge_base.merge_and_unload()
_merge_base.save_pretrained(MERGED_SFT_DIR)
del _merge_base
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
print(f"Merged model saved to {MERGED_SFT_DIR}")

# ── Step B: Reload merged model in 4-bit QLoRA ───────────────────────────────
grpo_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=os.environ["HF_TOKEN"])
grpo_tokenizer.pad_token = grpo_tokenizer.eos_token
grpo_tokenizer.padding_side = "left"

grpo_base = AutoModelForCausalLM.from_pretrained(
    MERGED_SFT_DIR,
    quantization_config=QNF4_CONFIG,
    device_map={"": 0},
)
grpo_lora = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
grpo_model = get_peft_model(grpo_base, grpo_lora)
grpo_model.print_trainable_parameters()

# ── Step C: Reload frozen RM in 4-bit (both models on GPU in 4-bit) ──────────
_rm_base2 = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_ID, num_labels=1,
    quantization_config=QNF4_CONFIG, device_map={"": 0},
    token=os.environ["HF_TOKEN"],
)
_rm_base2.config.pad_token_id = rm_tokenizer.pad_token_id
rm_frozen = PeftModel.from_pretrained(_rm_base2, RM_OUTPUT_DIR)
rm_frozen.eval()
for p in rm_frozen.parameters():
    p.requires_grad = False
print("Both models loaded in 4-bit. Ready for GRPO.")

In [ ]:
NUM_GENERATIONS = 2

grpo_config = GRPOConfig(
    output_dir=GRPO_OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=NUM_GENERATIONS,
    gradient_accumulation_steps=32,
    learning_rate=5e-6,
    max_prompt_length=256,
    max_completion_length=128,
    num_generations=NUM_GENERATIONS,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    bf16=True,
)

grpo_trainer = GRPOTrainer(
    model=grpo_model,
    args=grpo_config,
    train_dataset=grpo_dataset,
    reward_funcs=safety_reward_fn,
    processing_class=grpo_tokenizer,
)
grpo_trainer.train()

grpo_model.save_pretrained(GRPO_OUTPUT_DIR)
grpo_tokenizer.save_pretrained(GRPO_OUTPUT_DIR)
print("GRPO adapter saved to", GRPO_OUTPUT_DIR)

In [ ]:
# ── Zip outputs for download ──────────────────────────────────────────────────
shutil.make_archive("/kaggle/working/rm_adapter",   "zip", "/kaggle/working/outputs/rm")
shutil.make_archive("/kaggle/working/grpo_adapter", "zip", "/kaggle/working/outputs/grpo")
print("Done! Download rm_adapter.zip and grpo_adapter.zip from the Output tab.")